1. What percentage of all leads with **Online Application** have been scored by **both** systems? 27.55%
2. What percentage of **dually scored** leads receive the **same qualitative outcome** (high-quality / low-quality) from both systems? 73.33%
3. How do the two scoring systems compare in terms of the **absolute amount** of high-quality leads, split by the two main marketing channels: **Meta** and **Google**?

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "Case Study - Growth Data Analyst.xlsx"

df = pd.read_excel(DATA_PATH)
df.shape, df.columns.tolist()


((1865, 38),
 ['Level 1',
  'Level 2',
  'academy',
  'country',
  'date_type',
  'formname',
  'Marketing Channel',
  'Marketing Source',
  'product_lead',
  'reg_method',
  'UTM-campaign',
  'UTM-content/term',
  'UTM-medium',
  'UTM-source',
  'UTM-term/content',
  'Creation Date',
  'first_enrolled_status',
  'interest_level',
  'Lead ID',
  'Marketing channel NOT PROCESSED',
  'UTM-content',
  'UTM-term',
  'CR',
  'L1, %',
  'L2, %',
  'L3, %',
  'L4, %',
  'Leads',
  'Leads FIlter',
  'Qual',
  'Qual %',
  'Qual % FIlter',
  'UnQual %',
  'A, %',
  'B, %',
  'C, %',
  'D, %',
  'E, %'])

We check key columns used in this analysis:
- `reg_method` (scope filter)
- ABCDE scoring (A–E) — derived from `%` columns in the export: `A, %` ... `E, %`
- L1234 scoring (L1–L4) — in this export it is represented by `interest_level` (values 1–4)
- marketing channel — `Marketing Channel` (contains e.g. *Google Ads*, *Facebook Ads*)


In [3]:
df[['reg_method','interest_level','Marketing Channel','A, %','B, %','C, %','D, %','E, %']].head(10)


,reg_method,interest_level,Marketing Channel,"A, %","B, %","C, %","D, %","E, %"
0,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
1,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
2,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
3,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
4,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
5,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
6,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
7,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
8,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0
9,Application via Test,NaN,Google Ads,0.0,0.0,0.0,0.0,0.0


### ABCDE
The export provides percent columns (`A, %` ... `E, %`).  
For each lead, we assign the **first letter** whose percent is greater than 0.

### L1234
The manual qualification is represented by `interest_level` (1–4), which we treat as:
- L1/L2 = high-quality
- L3/L4 = low-quality


In [4]:
def derive_abcd_label(row):
    for letter in ['A','B','C','D','E']:
        if pd.notna(row.get(f'{letter}, %')) and row[f'{letter}, %'] > 0:
            return letter
    return np.nan

df = df.copy()
df['ABCDE'] = df.apply(derive_abcd_label, axis=1)
df['L1234'] = df['interest_level']  # numeric 1..4 (may be NaN)

df[['ABCDE','L1234']].head(10), df[['ABCDE','L1234']].isna().mean()


(  ABCDE  L1234
 0   NaN    NaN
 1   NaN    NaN
 2   NaN    NaN
 3   NaN    NaN
 4   NaN    NaN
 5   NaN    NaN
 6   NaN    NaN
 7   NaN    NaN
 8   NaN    NaN
 9   NaN    NaN,
 ABCDE    0.730831
 L1234    0.704021
 dtype: float64)

Definition:
- Scope: `reg_method == "Online Application"`
- Dually scored: both `ABCDE` and `L1234` are present (non-null)


In [5]:
online = df[df['reg_method'] == "Online Application"].copy()

dually_scored = online.dropna(subset=['ABCDE','L1234']).copy()

q1_pct = 100 * len(dually_scored) / len(online) if len(online) else np.nan

{
    "online_application_leads": int(len(online)),
    "dually_scored_leads": int(len(dually_scored)),
    "pct_dually_scored": round(q1_pct, 2)
}


{'online_application_leads': 871,
 'dually_scored_leads': 240,
 'pct_dually_scored': 27.55}

Qualitative mapping:
- **ABCDE**: A/B = High-quality (HQ), C/D/E = Low-quality (LQ)
- **L1234**: 1/2 = High-quality (HQ), 3/4 = Low-quality (LQ)

We compute the share of dually scored leads where HQ/LQ matches between the two systems.


In [6]:
def qual_from_abcd(x):
    return "HQ" if x in ["A","B"] else "LQ"

def qual_from_l1234(x):
    return "HQ" if x in [1,2] else "LQ"

dually_scored['qual_abcd'] = dually_scored['ABCDE'].apply(qual_from_abcd)
dually_scored['qual_l1234'] = dually_scored['L1234'].apply(qual_from_l1234)
dually_scored['qual_match'] = dually_scored['qual_abcd'] == dually_scored['qual_l1234']

q2_pct = 100 * dually_scored['qual_match'].mean() if len(dually_scored) else np.nan

{
    "dually_scored_leads": int(len(dually_scored)),
    "pct_same_qualitative_outcome": round(q2_pct, 2)
}


{'dually_scored_leads': 240, 'pct_same_qualitative_outcome': 73.33}

This helps understand *where* disagreements happen (e.g., ABCDE says HQ but sales says LQ).


In [7]:
pd.crosstab(dually_scored['qual_abcd'], dually_scored['qual_l1234'], margins=True)

qual_l1234,HQ,LQ,All
qual_abcd,,,
HQ,31,50,81
LQ,14,145,159
All,45,195,240


We focus on the main channels:
- **Google Ads**
- **Facebook Ads** (Meta)

For each channel, we count high-quality leads according to:
- ABCDE (A/B)
- L1234 (1/2)

Note: This question is about **absolute counts**, so we report counts rather than percentages.


In [8]:
main_channels = ["Google Ads", "Facebook Ads"]

d_main = dually_scored[dually_scored['Marketing Channel'].isin(main_channels)].copy()

summary = (
    d_main
    .groupby('Marketing Channel', as_index=False)
    .agg(
        leads_dually_scored=('Lead ID', 'count'),
        hq_abcd=('ABCDE', lambda s: int(s.isin(['A','B']).sum())),
        hq_l1234=('L1234', lambda s: int(pd.Series(s).isin([1,2]).sum()))
    )
)

summary


,Marketing Channel,leads_dually_scored,hq_abcd,hq_l1234
0,Facebook Ads,39,19,15
1,Google Ads,174,47,17


Final metrics for reporting.


In [9]:
answers = {}

answers["Q1_pct_online_application_scored_by_both"] = round(q1_pct, 2)
answers["Q2_pct_dually_scored_with_same_qual_outcome"] = round(q2_pct, 2)

# Channel table for Q3
q3_table = summary.c

answers, q3_table


({'Q1_pct_online_application_scored_by_both': 27.55,
  'Q2_pct_dually_scored_with_same_qual_outcome': 73.33},
   Marketing Channel  leads_dually_scored  hq_abcd  hq_l1234
 0      Facebook Ads                   39       19        15
 1        Google Ads                  174       47        17)

### Interpretation notes (optional)

- **Coverage issue:** If only a minority of Online Application leads are scored by both systems, optimization based on “quality” may be biased toward reachable/scorable leads.
- **Alignment:** A ~70%+ qualitative match indicates reasonable agreement, but disagreements are still material and worth investigating (question design, sales process, timing).
- **Channel comparison:** Differences between ABCDE vs L1234 high-quality counts can suggest that automatic “form quality” does not fully predict sales-qualified quality, especially by channel.
